# Bengaluru Traffic Digital Twin — Inference & Frontend Notebook
Loads the trained models, runs zero-shot cross-zone evaluation, computes
MAE/RMSE/MAPE/degradation and topology-distance correlations, generates all
visualizations, and assembles the data bundle consumed by the interactive
`digital_twin.html` Digital Twin frontend.

In [1]:
import sys, json
sys.path.insert(0, "/home/claude/bengaluru-digital-twin")
import pandas as pd, numpy as np
from topology.extract_topology import load_zone_graphs, build_feature_table, topology_distance_matrix
from evaluation.run_experiment import (
    train_model_family, zero_shot_eval, compute_degradation,
    topo_distance_to_degradation, TRAIN_ZONES, TEST_ZONES,
)

In [2]:
graphs = load_zone_graphs("/home/claude/bengaluru-digital-twin/data/raw/bengaluru_zone_graphs.json")
with open("/home/claude/bengaluru-digital-twin/data/raw/bengaluru_traffic_signals.json") as f:
    sig_data = json.load(f)
signals = sig_data["signals"]
feat_df = build_feature_table(graphs)
dist_df = topology_distance_matrix(feat_df)

trained, seen_metrics = train_model_family(graphs, signals, lookback=8, epochs=40)
seen_metrics.groupby("model")[["MAE","RMSE","MAPE"]].mean().round(3)

/home/claude/bengaluru-digital-twin/models/gnn_models.py:32: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  d_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


,MAE,RMSE,MAPE
model,,,
AdaptiveGNN,101.210,126.914,22.094
BaselineGCN,98.075,123.380,21.874
TopologyAwareAGNN,99.385,124.792,21.650


## Zero-shot cross-zone evaluation

In [3]:
zs_metrics = zero_shot_eval(trained, graphs, signals, lookback=8)
degradation = compute_degradation(seen_metrics, zs_metrics)
degradation = topo_distance_to_degradation(degradation, dist_df)
degradation.round(3)

/home/claude/bengaluru-digital-twin/models/gnn_models.py:32: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  d_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


,model,test_zone,MAE,RMSE,MAPE,seen_MAE,seen_RMSE,seen_MAPE,MAE_degradation_pct,RMSE_degradation_pct,topo_distance_from_train
0,AdaptiveGNN,ElectronicCity,77.475,97.001,22.270,101.210,126.914,22.094,-23.451,-23.569,5.165
1,AdaptiveGNN,Whitefield,79.694,100.115,21.159,101.210,126.914,22.094,-21.259,-21.115,4.380
2,AdaptiveGNN,Yelahanka,66.641,83.844,20.982,101.210,126.914,22.094,-34.156,-33.937,5.711
3,BaselineGCN,ElectronicCity,75.791,94.811,22.460,98.075,123.380,21.874,-22.721,-23.155,5.165
4,BaselineGCN,Whitefield,79.359,99.573,21.688,98.075,123.380,21.874,-19.084,-19.296,4.380
5,BaselineGCN,Yelahanka,70.562,88.300,22.920,98.075,123.380,21.874,-28.053,-28.433,5.711
6,TopologyAwareAGNN,ElectronicCity,74.143,92.991,21.387,99.385,124.792,21.650,-25.398,-25.483,5.165
7,TopologyAwareAGNN,Whitefield,77.385,97.356,20.579,99.385,124.792,21.650,-22.137,-21.985,4.380
8,TopologyAwareAGNN,Yelahanka,65.458,82.108,20.838,99.385,124.792,21.650,-34.137,-34.204,5.711


## Topology-distance vs degradation correlation

In [4]:
corr_rows = []
for model in degradation["model"].unique():
    sub_ = degradation[degradation["model"] == model]
    r = np.corrcoef(sub_["topo_distance_from_train"], sub_["MAE_degradation_pct"])[0, 1]
    corr_rows.append({"model": model, "pearson_r_topo_vs_degradation": r})
corr_df = pd.DataFrame(corr_rows)
corr_df.round(3)

,model,pearson_r_topo_vs_degradation
0,AdaptiveGNN,-0.893
1,BaselineGCN,-0.978
2,TopologyAwareAGNN,-0.936


## Generate all visualizations

In [5]:
zs_metrics.to_csv("/home/claude/bengaluru-digital-twin/evaluation/zero_shot_raw_metrics.csv", index=False)
degradation.to_csv("/home/claude/bengaluru-digital-twin/evaluation/degradation_summary.csv", index=False)
corr_df.to_csv("/home/claude/bengaluru-digital-twin/evaluation/topology_degradation_correlation.csv", index=False)

import subprocess
subprocess.run(["python3", "/home/claude/bengaluru-digital-twin/visualizations/make_plots.py"], check=True)
print("Plots written to visualizations/")

/home/claude/bengaluru-digital-twin/visualizations/make_plots.py:131: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sub["zone"], rotation=45, ha="right")


all plots written to /home/claude/bengaluru-digital-twin/visualizations
Plots written to visualizations/


## Frontend data bundle
The interactive Digital Twin (`frontend/digital_twin.html`) is a self-contained
HTML file with the graph/topology/signal/results data inlined as JSON at build
time (see `frontend/assets/twin_data.json` and the assembly step in the
project README). Re-run the assembly step after changing any upstream data to
refresh the frontend bundle.

In [6]:
print("Zones:", list(graphs.keys()))
print("Train zones:", TRAIN_ZONES)
print("Test (zero-shot) zones:", TEST_ZONES)
print("\nOpen frontend/digital_twin.html to explore the live Digital Twin.")

Zones: ['Koramangala', 'SilkBoard', 'Indiranagar', 'MGRoad_CBD', 'Whitefield', 'ElectronicCity', 'Hebbal', 'Yelahanka', 'Jayanagar', 'Malleshwaram']
Train zones: ['Koramangala', 'SilkBoard', 'MGRoad_CBD', 'Indiranagar', 'Jayanagar', 'Malleshwaram', 'Hebbal']
Test (zero-shot) zones: ['Whitefield', 'ElectronicCity', 'Yelahanka']

Open frontend/digital_twin.html to explore the live Digital Twin.
